# 05 — Middleware & Context Engineering

## Learning requirements
- hiểu middleware hooks quanh agent/model/tool lifecycle;
- dynamic prompt/model/tool selection;
- summarization/context trimming;
- context isolation;
- thiết kế context theo **who needs what, when, why**.

Nhiều agent failure không phải vì model yếu mà vì context sai, thừa, thiếu hoặc không đúng thời điểm.

## Context engineering framework

Trước mỗi model call, trả lời:

1. Model cần biết gì để quyết định bước này?
2. Context đó đến từ system instruction, messages, retrieval, tool result hay store?
3. Nó có chứa dữ liệu user/tenant khác không?
4. Có thể summarize/offload không?
5. Có tool nào không liên quan nên ẩn đi không?
6. Có output nào quá lớn cần truncate/store ngoài context không?

```text
MODEL CONTEXT
  system prompt
  messages
  available tools
  retrieved evidence
  response schema

TOOL CONTEXT
  runtime identity
  thread state
  persistent store

LIFECYCLE CONTEXT
  logging
  summarization
  guardrails
  retries
```

In [ ]:
# Built-in summarization middleware example.
# Tune thresholds to your provider/model context window.
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from pathlib import Path
import sys
root = Path.cwd()
while not (root / "requirements.txt").exists() and root.parent != root:
    root = root.parent
sys.path.insert(0, str(root))
from src.providers import get_chat_model

model = get_chat_model()

agent = create_agent(
    model=model,
    tools=[],
    middleware=[
        SummarizationMiddleware(
            model=model,
            trigger=("tokens", 4000),
            keep=("messages", 20),
        )
    ],
)

## Middleware topics to investigate

- before/after agent
- before/after model
- wrap model call
- wrap tool call
- retry/fallback
- dynamic system prompt
- dynamic model routing
- dynamic tool exposure
- PII/guardrail middleware
- human-in-the-loop middleware

API chi tiết có thể đổi; học lifecycle + responsibility trước syntax.

In [ ]:
# Simple model routing logic outside middleware first.
# Then move this idea into middleware when comfortable.
def choose_model_name(task_complexity: str) -> str:
    if task_complexity == "simple":
        return "fast-cheap-model"
    return "stronger-model"

for c in ["simple", "complex"]:
    print(c, "->", choose_model_name(c))

## Required output — `artifacts/context-engineering.md`

Document phải trả lời cho một Interview Agent:

- context nào scanner agent thấy;
- context nào interviewer thấy;
- context nào validator thấy;
- dữ liệu nào chỉ thuộc thread;
- dữ liệu nào lưu cross-thread;
- tool outputs nào phải offload;
- khi nào summarize;
- token budget;
- tenant/user isolation;
- prompt-injection boundary.

## Done criteria
Bạn không còn giải quyết mọi vấn đề bằng cách “thêm hết dữ liệu vào system prompt”.